# Neat Tensor

This notebook shows how a real image loaded as a NumPy array becomes a Neat tensor and maps back to NumPy.


## Import Packages

In [1]:
from pathlib import Path
import numpy as np
import cv2
import pyneat


## Load Image

In [2]:
IMAGE_PATH = Path("../assets/images/image.png")
image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
print("image:", IMAGE_PATH, image_bgr.shape, image_bgr.dtype)


image: ../assets/images/image.png (563, 1000, 3) uint8


## BGR and RGB Image Tensors

OpenCV reads BGR images. Tell Neat the format you are providing.

Pyneat BGR Tensor

In [3]:
t_bgr = pyneat.Tensor.from_numpy(image_bgr, 
                                 copy=True, 
                                 image_format=pyneat.PixelFormat.BGR)

bgr_back = t_bgr.to_numpy(copy=True)

print("BGR tensor:", tuple(t_bgr.shape), "same:", bool(np.array_equal(image_bgr, bgr_back)))

BGR tensor: (563, 1000, 3) same: True


Pyneat RGB Tensor

In [4]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

t_rgb = pyneat.Tensor.from_numpy(image_rgb, 
                                 copy=True, 
                                 image_format=pyneat.PixelFormat.RGB)

rgb_back = t_rgb.to_numpy(copy=True)

print("RGB tensor:", tuple(t_rgb.shape), "same:", bool(np.array_equal(image_rgb, rgb_back)))


RGB tensor: (563, 1000, 3) same: True


## Inspect The Image Tensor

A Neat tensor has more metadata than only `shape`. For an image tensor, these fields describe the numeric type, memory layout, device placement, image semantic, storage, and whether the image uses packed or multi-plane formats.

In [5]:
print("shape:", tuple(t_rgb.shape))
print("dtype:", t_rgb.dtype)
print("layout:", t_rgb.layout)
print("device:", t_rgb.device)
print("semantic:", t_rgb.semantic)
print("strides_bytes:", t_rgb.strides_bytes)
print("byte_offset:", t_rgb.byte_offset)
print("read_only:", t_rgb.read_only)
print("storage:", t_rgb.storage)

shape: (563, 1000, 3)
dtype: TensorDType.UInt8
layout: TensorLayout.HWC
device: <pyneat._pyneat_core.Device object at 0xffff11198250>
semantic: <pyneat._pyneat_core.Semantic object at 0xffff11198270>
strides_bytes: [3000, 3, 1]
byte_offset: 0
read_only: False
storage: <pyneat._pyneat_core.Storage object at 0xffff11198270>


In [6]:
print("width:", t_rgb.width())
print("height:", t_rgb.height())
print("channels:", t_rgb.channels())
print("is_nv12:", t_rgb.is_nv12())
print("is_i420:", t_rgb.is_i420())

width: 1000
height: 563
channels: 3
is_nv12: False
is_i420: False


In [7]:
print("is_dense:", t_rgb.is_dense())
print("is_contiguous:", t_rgb.is_contiguous())
print("is_composite:", t_rgb.is_composite())

is_dense: True
is_contiguous: True
is_composite: False


## Non-Image Tensor Payload

Use regular `Tensor.from_numpy` for numeric arrays that are not images.

In [8]:
features = image_bgr.astype(np.float32) / 255.0
feature_tensor = pyneat.Tensor.from_numpy(features, copy=True)

print("feature tensor shape:", tuple(feature_tensor.shape))
print("roundtrip dtype:", feature_tensor.to_numpy(copy=True).dtype)


feature tensor shape: (563, 1000, 3)
roundtrip dtype: float32


## Copy Behavior

`copy=True` asks Neat to keep its own safe copy of the NumPy data.

`copy=False` gives Neat permission to avoid a copy when that is safe for the current tensor layout, memory type, and runtime path. It is not a guarantee that NumPy memory will be shared. The runtime may still copy internally, so application code should not depend on mutations to the original NumPy array being visible through the tensor.


In [9]:
scratch = np.zeros((4, 4, 3), dtype=np.uint8)
scratch[..., 0] = 10

t_copy_true = pyneat.Tensor.from_numpy(
    scratch,
    copy=True,
    image_format=pyneat.PixelFormat.RGB,
)

t_copy_false = pyneat.Tensor.from_numpy(
    scratch,
    copy=False,
    image_format=pyneat.PixelFormat.RGB,
)

scratch[..., 0] = 99

copy_true_back = t_copy_true.to_numpy(copy=True)
copy_false_back = t_copy_false.to_numpy(copy=True)

print("original NumPy first pixel after edit:", scratch[0, 0].tolist())
print("copy=True tensor first pixel:", copy_true_back[0, 0].tolist())
print("copy=False tensor first pixel:", copy_false_back[0, 0].tolist())
print("copy=False shared original mutation:", bool(copy_false_back[0, 0, 0] == scratch[0, 0, 0]))


original NumPy first pixel after edit: [99, 0, 0]
copy=True tensor first pixel: [10, 0, 0]
copy=False tensor first pixel: [10, 0, 0]
copy=False shared original mutation: False


## `to_numpy` Copy Behavior

`to_numpy(copy=True)` returns a safe NumPy copy. `to_numpy(copy=False)` may return a view or borrowed buffer when possible. Use `copy=True` when you plan to modify or store the array.

In [10]:
rgb_copy = t_rgb.to_numpy(copy=True)
rgb_view = t_rgb.to_numpy(copy=False)

print("copy shape/dtype:", rgb_copy.shape, rgb_copy.dtype)
print("no-copy shape/dtype:", rgb_view.shape, rgb_view.dtype)
print("same pixels:", bool(np.array_equal(rgb_copy, rgb_view)))


copy shape/dtype: (563, 1000, 3) uint8
no-copy shape/dtype: (563, 1000, 3) uint8
same pixels: True


## HWC Layout When Passing NumPy Directly

When you pass a NumPy image directly into a run or push API, tell Neat both the memory layout and the pixel format. The array shape says there are three channels; `PixelFormat` says what those channels mean.

In [11]:
print("layout for OpenCV-style image arrays:", pyneat.TensorLayout.HWC)
print("RGB pixel format:", pyneat.PixelFormat.RGB)
print("BGR pixel format:", pyneat.PixelFormat.BGR)

# Example usage in graph/model execution code:
# run.push([image_rgb], layout=pyneat.TensorLayout.HWC, image_format=pyneat.PixelFormat.RGB)


layout for OpenCV-style image arrays: TensorLayout.HWC
RGB pixel format: PixelFormat.RGB
BGR pixel format: PixelFormat.BGR


## Tensor Inside A Sample

A `Tensor` is the image/data payload. A `Sample` wraps that payload with stream metadata such as stream id, frame id, and timestamp.

In [12]:
sample = pyneat.Sample()
sample.kind = pyneat.SampleKind.Tensor
sample.tensor = t_rgb
sample.stream_id = "tutorial"
sample.frame_id = 1
sample.pts_ns = 0

print("sample kind:", sample.kind)
print("stream/frame:", sample.stream_id, sample.frame_id)
print("has tensor:", sample.tensor is not None)


sample kind: SampleKind.Tensor
stream/frame: tutorial 1
has tensor: True


## Reading Tensor Outputs From Samples

Some graph outputs store the tensor at `sample.tensor`. Others store tensors in `sample.tensors`, especially when the output is a tensor set. A tiny helper keeps the code robust.

In [13]:
def first_tensor(sample):
    if sample is None:
        raise RuntimeError("sample is None")
    if sample.tensor is not None:
        return sample.tensor
    if sample.tensors:
        return sample.tensors[0]
    raise RuntimeError("sample does not contain a tensor")

single_tensor = pyneat.Sample()
single_tensor.kind = pyneat.SampleKind.Tensor
single_tensor.tensor = t_rgb

tensor_set = pyneat.Sample()
tensor_set.kind = pyneat.SampleKind.TensorSet
tensor_set.tensors = [t_rgb]

print("single tensor shape:", tuple(first_tensor(single_tensor).shape))
print("tensor set shape:", tuple(first_tensor(tensor_set).shape))


single tensor shape: (563, 1000, 3)
tensor set shape: (563, 1000, 3)


## References

- Public tutorial: [pass NumPy to model](https://developer.sima.ai/software/tutorials/pass-numpy-to-model).
- Core source: [pass_numpy_to_model.py](https://github.com/sima-neat/core/blob/main/tutorials/009_pass_numpy_to_model/pass_numpy_to_model.py).